# 04 — Narrative Velocity & Campaign PredictionHow fast do narratives spread? What predicts a campaign?**Formulas derived:**- Velocity: $v(t) = \frac{n(t) - n(t-w)}{w}$ (3-day smoothed)- Spread: $s(t) = |\{\text{source types carrying narrative}\}|$- Amplification: $A = \frac{\max(\text{burst})}{\text{burst}[0]}$- Campaign score: $CS(t) = \sum_{N_k} \max(0, v_k(t)) \cdot s_k(t)$

In [ ]:
import json, osimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsDATA = os.environ.get("ESTWARDEN_DATA", "../data")def load(f):    out = []    for line in open(f"{DATA}/{f}"):        try: out.append(json.loads(line.strip()))        except: pass    return outtags = load("narrative_tags.jsonl")campaigns = load("campaigns.jsonl")media = load("media_signals.jsonl")CODES = {"N1": "Russophobia", "N2": "War Panic", "N3": "Aid=Theft",         "N4": "Delegitimize", "N5": "Isolation"}df_tags = pd.DataFrame(tags)df_tags["date"] = pd.to_datetime(df_tags["created_at"]).dt.dateprint(f"Tags: {len(df_tags)}, Campaigns: {len(campaigns)}")

## Daily narrative volume

In [ ]:
daily_narr = df_tags.groupby(["date", "code"]).size().unstack(fill_value=0)daily_narr.index = pd.to_datetime(daily_narr.index)fig, ax = plt.subplots(figsize=(14, 5))daily_narr.plot.bar(stacked=True, ax=ax, colormap="Set2", width=0.9)ax.set_title("Daily Narrative Tag Volume by Code")ax.set_ylabel("Tags")ax.legend(title="Code")plt.tight_layout()plt.savefig("narrative_volume.png", dpi=150)plt.show()daily_narr.describe().round(1)

## Narrative velocity (3-day smoothed Δ/day)

In [ ]:
window = 3velocity = daily_narr.diff(window) / windowfig, ax = plt.subplots(figsize=(14, 5))velocity.plot(ax=ax, marker="o", markersize=3)ax.axhline(0, color="black", linewidth=0.5)ax.set_title("Narrative Velocity (Δ tags/day, 3-day window)")ax.set_ylabel("Velocity")plt.tight_layout()plt.savefig("narrative_velocity.png", dpi=150)plt.show()print("Peak velocities:")for code in daily_narr.columns:    v = velocity[code].dropna()    if len(v) == 0: continue    peak_date = v.idxmax()    peak_val = v.max()    print(f"  {code} ({CODES.get(code,'')}): {peak_val:+.1f}/day on {peak_date.date()}")

## Burst detection & amplification ratio

In [ ]:
print("Narrative bursts:\n")burst_data = []for code in daily_narr.columns:    series = daily_narr[code]    # Burst = consecutive days with count > 0    in_burst = False    start = None    for date, val in series.items():        if val > 0 and not in_burst:            in_burst = True            start = date            burst_vals = [val]        elif val > 0 and in_burst:            burst_vals.append(val)        elif val == 0 and in_burst:            in_burst = False            if len(burst_vals) >= 2:                amp = max(burst_vals) / burst_vals[0]                burst_data.append({                    "code": code, "start": start, "duration": len(burst_vals),                    "first": burst_vals[0], "peak": max(burst_vals),                    "total": sum(burst_vals), "amplification": amp                })    # Handle ongoing burst    if in_burst and len(burst_vals) >= 2:        amp = max(burst_vals) / burst_vals[0]        burst_data.append({            "code": code, "start": start, "duration": len(burst_vals),            "first": burst_vals[0], "peak": max(burst_vals),            "total": sum(burst_vals), "amplification": amp        })df_bursts = pd.DataFrame(burst_data)if len(df_bursts):    print(df_bursts.sort_values("amplification", ascending=False).to_string(index=False))    print(f"\nMean amplification: {df_bursts['amplification'].mean():.1f}x")    print(f"Max amplification: {df_bursts['amplification'].max():.1f}x")

## Campaign prediction$$CS(t) = \sum_{k} \max(0,\; v_k(t)) \cdot s_k(t)$$Compare campaign score against known campaign detection dates.

In [ ]:
# Campaign datescampaign_dates = set()for c in campaigns:    cd = c.get("detected_at", "")[:10]    if cd: campaign_dates.add(pd.Timestamp(cd))# Campaign score = velocity × spread (approximated by number of active narratives)positive_vel = velocity.clip(lower=0)n_active = (daily_narr > 0).sum(axis=1)campaign_score = positive_vel.sum(axis=1) * n_activefig, ax = plt.subplots(figsize=(14, 5))colors = ["red" if d in campaign_dates else "steelblue" for d in campaign_score.index]ax.bar(range(len(campaign_score)), campaign_score.values, color=colors, width=0.9)ax.set_title("Campaign Score (red = known campaign detected)")ax.set_ylabel("CS(t)")plt.tight_layout()plt.savefig("campaign_score.png", dpi=150)plt.show()# Evaluate: does high CS predict campaign detection?from sklearn.metrics import roc_auc_scorey_campaign = np.array([1 if d in campaign_dates else 0 for d in campaign_score.index])if y_campaign.sum() > 0 and y_campaign.sum() < len(y_campaign):    cs_auc = roc_auc_score(y_campaign, campaign_score.fillna(0).values)    print(f"Campaign Score AUC: {cs_auc:.3f}")else:    print("Not enough campaign variation for AUC")

## Confidence vs accuracyAre higher-confidence classifications more likely to be correct?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))df_tags.boxplot(column="confidence", by="code", ax=ax)ax.set_title("Classification Confidence by Narrative Code")ax.set_ylabel("Confidence")plt.suptitle("")plt.tight_layout()plt.savefig("confidence_by_code.png", dpi=150)plt.show()print(f"\nConfidence stats:")print(df_tags.groupby("code")["confidence"].describe().round(3).to_string())